# Comprehensive Exploratory Data Analysis (EDA) Pipeline
## Automated Data Profiling & Reporting System

This notebook provides a **production-ready EDA workflow** that automatically:
- Loads data from multiple file formats
- Generates comprehensive profiling reports
- Exports structured summaries and statistics
- Organizes outputs with metadata tracking

**Supported Formats:** CSV, Excel, JSON, Parquet, Feather, Pickle, TSV

---

## 1. Setup & Configuration

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import sys
import warnings
import logging
from pathlib import Path
from datetime import datetime
import chardet
from io import StringIO
import traceback

warnings.filterwarnings('ignore')

print('✓ Core libraries imported successfully')

## 2. Configuration & Paths

In [ ]:
# ============================================
# CONFIGURATION SECTION - MODIFY AS NEEDED
# ============================================

DATA_FILE = 'your_dataset.csv'  # Change this to your file path

BASE_OUTPUT_DIR = 'reports'
MAX_ROWS_DISPLAY = 100
CORRELATION_MIN_THRESHOLD = 0.5

GENERATE_YDATA_PROFILE = True
GENERATE_SWEETVIZ = True
GENERATE_AUTOVIZ = True
LAUNCH_DTALE = True

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print(f'Data file: {DATA_FILE}')
print(f'Output directory: {BASE_OUTPUT_DIR}')

## 3. Logging Setup

In [ ]:
Path(BASE_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

log_file = os.path.join(BASE_OUTPUT_DIR, 'eda_log.txt')
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_file),
        logging.StreamHandler()
    ]
)

logger = logging.getLogger(__name__)
logger.info('='*80)
logger.info('EDA Pipeline Started')
logger.info(f'Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
logger.info('='*80)

print(f'✓ Logging initialized. Log file: {log_file}')

## 4. Utility Functions

In [ ]:
def detect_encoding(file_path, sample_size=10000):
    try:
        with open(file_path, 'rb') as f:
            raw_data = f.read(sample_size)
        result = chardet.detect(raw_data)
        encoding = result['encoding'] or 'utf-8'
        logger.info(f'Detected encoding: {encoding}')
        return encoding
    except Exception as e:
        logger.warning(f'Encoding detection failed: {e}. Using utf-8.')
        return 'utf-8'

def detect_delimiter(file_path, encoding):
    try:
        with open(file_path, 'r', encoding=encoding) as f:
            sample = f.read(10000)
        delimiters = [',', ';', '\t', '|', ':']
        delimiter_counts = {delim: sample.count(delim) for delim in delimiters}
        detected_delim = max(delimiter_counts, key=delimiter_counts.get)
        logger.info(f'Detected delimiter: {detected_delim}')
        return detected_delim
    except Exception as e:
        logger.warning(f'Delimiter detection failed: {e}. Using comma.')
        return ','

print('✓ Utility functions defined')

## 5. Data Loading Function

In [ ]:
def load_data(file_path):
    if not os.path.exists(file_path):
        logger.error(f'File not found: {file_path}')
        raise FileNotFoundError(f'Data file not found: {file_path}')
    
    file_ext = os.path.splitext(file_path)[1].lower()
    logger.info(f'Loading file: {file_path} (Type: {file_ext})')
    
    try:
        if file_ext in ['.csv', '.tsv']:
            encoding = detect_encoding(file_path)
            delimiter = detect_delimiter(file_path, encoding)
            df = pd.read_csv(file_path, encoding=encoding, sep=delimiter)
            logger.info(f'CSV loaded successfully')
        elif file_ext in ['.xlsx', '.xls']:
            df = pd.read_excel(file_path)
            logger.info('Excel file loaded successfully')
        elif file_ext == '.json':
            df = pd.read_json(file_path)
            logger.info('JSON file loaded successfully')
        elif file_ext == '.parquet':
            df = pd.read_parquet(file_path)
            logger.info('Parquet file loaded successfully')
        elif file_ext == '.feather':
            df = pd.read_feather(file_path)
            logger.info('Feather file loaded successfully')
        elif file_ext in ['.pkl', '.pickle']:
            df = pd.read_pickle(file_path)
            logger.info('Pickle file loaded successfully')
        else:
            raise ValueError(f'Unsupported file format: {file_ext}')
        return df
    except Exception as e:
        logger.error(f'Error loading data: {e}')
        raise

print('✓ Data loading function defined')

## 6. Load Dataset

In [ ]:
start_time = datetime.now()

try:
    df = load_data(DATA_FILE)
    logger.info(f'Dataset loaded. Shape: {df.shape}')
    print(f'✓ Data loaded successfully!')
    print(f'  Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
except Exception as e:
    print(f'✗ Failed to load data: {e}')
    logger.error(f'Data loading failed: {e}')
    sys.exit(1)

## 7. Create Dataset-Specific Output Directory

In [ ]:
dataset_name = os.path.splitext(os.path.basename(DATA_FILE))[0]
dataset_output_dir = os.path.join(BASE_OUTPUT_DIR, dataset_name)
Path(dataset_output_dir).mkdir(parents=True, exist_ok=True)

logger.info(f'Output directory created: {dataset_output_dir}')
print(f'✓ Output directory: {dataset_output_dir}')

## 8. Dataset Overview

In [ ]:
print('\n' + '='*80)
print('DATASET OVERVIEW')
print('='*80)
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB')
print(f'\nColumns ({len(df.columns)}):') 
print(df.dtypes)
print('\nFirst few rows:')
print(df.head())

logger.info(f'Dataset shape: {df.shape}')

## 9. Generate Data Types Summary

In [ ]:
print('\n' + '='*80)
print('DATA TYPES SUMMARY')
print('='*80)

dtypes_summary = pd.DataFrame({
    'Column': df.columns,
    'Data Type': df.dtypes.values,
    'Non-Null Count': df.count().values,
    'Null Count': df.isnull().sum().values,
    'Null %': (df.isnull().sum().values / len(df) * 100).round(2)
})

dtypes_summary_file = os.path.join(dataset_output_dir, '01_data_types_summary.csv')
dtypes_summary.to_csv(dtypes_summary_file, index=False)

print(dtypes_summary)
print(f'\n✓ Data types summary saved: {dtypes_summary_file}')
logger.info(f'Data types summary exported to {dtypes_summary_file}')

## 10. Missing Values Analysis

In [ ]:
print('\n' + '='*80)
print('MISSING VALUES ANALYSIS')
print('='*80)

missing_summary = pd.DataFrame({
    'Column': df.columns,
    'Missing Count': df.isnull().sum().values,
    'Missing %': (df.isnull().sum().values / len(df) * 100).round(2)
}).sort_values('Missing Count', ascending=False)

missing_summary_file = os.path.join(dataset_output_dir, '02_missing_values.csv')
missing_summary.to_csv(missing_summary_file, index=False)

print(missing_summary)
total_missing = df.isnull().sum().sum()
print(f'\nTotal missing values: {total_missing}')
print(f'✓ Missing values summary saved: {missing_summary_file}')
logger.info(f'Missing values summary exported to {missing_summary_file}')

## 11. Numerical Statistics

In [ ]:
print('\n' + '='*80)
print('NUMERICAL STATISTICS')
print('='*80)

numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

if numerical_cols:
    numerical_stats = df[numerical_cols].describe(include='all').T
    numerical_stats['skew'] = df[numerical_cols].skew()
    numerical_stats['kurtosis'] = df[numerical_cols].kurtosis()
    
    numerical_stats_file = os.path.join(dataset_output_dir, '03_numerical_statistics.csv')
    numerical_stats.to_csv(numerical_stats_file)
    
    print(numerical_stats.round(4))
    print(f'\n✓ Numerical statistics saved: {numerical_stats_file}')
    logger.info(f'Numerical statistics exported to {numerical_stats_file}')
else:
    print('No numerical columns found in dataset.')

## 12. Categorical Statistics

In [ ]:
print('\n' + '='*80)
print('CATEGORICAL STATISTICS')
print('='*80)

categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()

if categorical_cols:
    categorical_stats = pd.DataFrame({
        'Column': categorical_cols,
        'Unique Count': [df[col].nunique() for col in categorical_cols],
        'Top Value': [df[col].value_counts().index[0] if len(df[col].value_counts()) > 0 else None for col in categorical_cols],
        'Top Value Frequency': [df[col].value_counts().values[0] if len(df[col].value_counts()) > 0 else 0 for col in categorical_cols]
    })
    
    categorical_stats_file = os.path.join(dataset_output_dir, '04_categorical_statistics.csv')
    categorical_stats.to_csv(categorical_stats_file, index=False)
    
    print(categorical_stats)
    print(f'\n✓ Categorical statistics saved: {categorical_stats_file}')
    logger.info(f'Categorical statistics exported to {categorical_stats_file}')
else:
    print('No categorical columns found in dataset.')

## 13. Correlation Matrix

In [ ]:
print('\n' + '='*80)
print('CORRELATION ANALYSIS')
print('='*80)

if len(numerical_cols) > 1:
    correlation_matrix = df[numerical_cols].corr()
    
    correlation_file = os.path.join(dataset_output_dir, '05_correlation_matrix.csv')
    correlation_matrix.to_csv(correlation_file)
    
    print('\nFull Correlation Matrix:')
    print(correlation_matrix.round(3))
    
    print(f'\nHigh Correlations (abs > {CORRELATION_MIN_THRESHOLD}):')
    high_corr = []
    for i in range(len(correlation_matrix.columns)):
        for j in range(i+1, len(correlation_matrix.columns)):
            if abs(correlation_matrix.iloc[i, j]) > CORRELATION_MIN_THRESHOLD:
                high_corr.append({
                    'Column 1': correlation_matrix.columns[i],
                    'Column 2': correlation_matrix.columns[j],
                    'Correlation': correlation_matrix.iloc[i, j]
                })
    
    if high_corr:
        high_corr_df = pd.DataFrame(high_corr)
        print(high_corr_df)
    else:
        print(f'No correlations found above {CORRELATION_MIN_THRESHOLD} threshold.')
    
    print(f'\n✓ Correlation matrix saved: {correlation_file}')
    logger.info(f'Correlation matrix exported to {correlation_file}')
else:
    print('Not enough numerical columns for correlation analysis.')

## 14. Generate YData Profiling Report

In [ ]:
if GENERATE_YDATA_PROFILE:
    print('\n' + '='*80)
    print('GENERATING YDATA PROFILING REPORT')
    print('='*80)
    
    try:
        from ydata_profiling import ProfileReport
        
        logger.info('Starting YData Profiling...')
        print('\nGenerating profile (this may take a moment)...')
        
        profile = ProfileReport(
            df,
            title=f'YData Profile Report - {dataset_name}',
            explorative=True,
            progress_bar=True
        )
        
        ydata_file = os.path.join(dataset_output_dir, '06_ydata_profiling_report.html')
        profile.to_file(ydata_file)
        
        print(f'✓ YData Profiling report saved: {ydata_file}')
        logger.info(f'YData Profiling report saved to {ydata_file}')
    
    except ImportError:
        print('⚠ YData Profiling not installed. Skipping...')
        logger.warning('YData Profiling not installed.')
    
    except Exception as e:
        print(f'⚠ Error generating YData report: {e}')
        logger.error(f'YData Profiling error: {e}')

## 15. Generate Sweetviz Report

In [ ]:
if GENERATE_SWEETVIZ:
    print('\n' + '='*80)
    print('GENERATING SWEETVIZ REPORT')
    print('='*80)
    
    try:
        import sweetviz as sv
        
        logger.info('Starting Sweetviz report generation...')
        print('\nGenerating Sweetviz report...')
        
        my_report = sv.analyze(df, target_name=None)
        
        sweetviz_file = os.path.join(dataset_output_dir, '07_sweetviz_report.html')
        my_report.show(sweetviz_file, open_browser=False)
        
        print(f'✓ Sweetviz report saved: {sweetviz_file}')
        logger.info(f'Sweetviz report saved to {sweetviz_file}')
    
    except ImportError:
        print('⚠ Sweetviz not installed. Skipping...')
        logger.warning('Sweetviz not installed.')
    
    except Exception as e:
        print(f'⚠ Error generating Sweetviz report: {e}')
        logger.error(f'Sweetviz error: {e}')

## 16. Generate AutoViz Report

In [ ]:
if GENERATE_AUTOVIZ:
    print('\n' + '='*80)
    print('GENERATING AUTOVIZ REPORT')
    print('='*80)
    
    try:
        from autoviz.AutoViz_Class import AutoViz_Class
        
        logger.info('Starting AutoViz visualization...')
        print('\nGenerating AutoViz visualizations...')
        
        autoviz_output_dir = os.path.join(dataset_output_dir, 'autoviz_plots')
        Path(autoviz_output_dir).mkdir(parents=True, exist_ok=True)
        
        AV = AutoViz_Class()
        dft = AV.fit_transform(
            DATA_FILE,
            sep=',',
            depVar='',
            dfte=None,
            header=0,
            verbose=0,
            lowess=False,
            chart_format='svg',
            max_rows_analyzed=len(df),
            max_cols_analyzed=len(df.columns)
        )
        
        print(f'✓ AutoViz plots saved to: {autoviz_output_dir}')
        logger.info(f'AutoViz plots saved to {autoviz_output_dir}')
    
    except ImportError:
        print('⚠ AutoViz not installed. Skipping...')
        logger.warning('AutoViz not installed.')
    
    except Exception as e:
        print(f'⚠ Error generating AutoViz plots: {e}')
        logger.error(f'AutoViz error: {e}')

## 17. Launch D-Tale Interactive Dashboard

In [ ]:
if LAUNCH_DTALE:
    print('\n' + '='*80)
    print('D-TALE INTERACTIVE DASHBOARD')
    print('='*80)
    
    try:
        import dtale
        
        logger.info('Launching D-Tale dashboard...')
        print('\nLaunching D-Tale interactive dashboard...\n')
        
        d = dtale.show(df, open_browser=False)
        dtale_url = d.open_browser()
        
        print(f'✓ D-Tale dashboard available at: {dtale_url}')
        print('\nNote: Keep this notebook running to access the D-Tale dashboard.')
        print('To stop D-Tale, call: d.kill()')
        
        logger.info(f'D-Tale dashboard launched at {dtale_url}')
        
        dtale_instance = d
    
    except ImportError:
        print('⚠ D-Tale not installed. Skipping...')
        logger.warning('D-Tale not installed.')
    
    except Exception as e:
        print(f'⚠ Error launching D-Tale: {e}')
        logger.error(f'D-Tale error: {e}')

## 18. Generate Text Summary

In [ ]:
print('\n' + '='*80)
print('GENERATING TEXT SUMMARY')
print('='*80)

summary_text = f'Dataset Analysis Summary - {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}\n'
summary_text += f'Dataset: {dataset_name}\n'
summary_text += f'Rows: {df.shape[0]:,}\n'
summary_text += f'Columns: {df.shape[1]}\n'
summary_text += f'Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB\n\n'

summary_text += 'COLUMNS:\n'
for col in df.columns:
    missing_count = df[col].isnull().sum()
    missing_pct = (missing_count / len(df)) * 100
    summary_text += f'{col:30} | Type: {str(df[col].dtype):15} | Missing: {missing_count:6} ({missing_pct:5.2f}%)\n'

if numerical_cols:
    summary_text += '\nNUMERICAL STATISTICS:\n'
    for col in numerical_cols:
        summary_text += f'{col}: min={df[col].min():.4f}, max={df[col].max():.4f}, mean={df[col].mean():.4f}\n'

if categorical_cols:
    summary_text += '\nCATEGORICAL STATISTICS:\n'
    for col in categorical_cols:
        top_value = df[col].value_counts().index[0]
        unique_count = df[col].nunique()
        summary_text += f'{col}: {unique_count} unique values, top={top_value}\n'

summary_file = os.path.join(dataset_output_dir, '00_summary.txt')
with open(summary_file, 'w') as f:
    f.write(summary_text)

print(summary_text)
print(f'\n✓ Text summary saved: {summary_file}')
logger.info(f'Text summary exported to {summary_file}')

## 19. Generate Manifest File

In [ ]:
print('\n' + '='*80)
print('GENERATING MANIFEST')
print('='*80)

generated_files = []
for item in os.listdir(dataset_output_dir):
    item_path = os.path.join(dataset_output_dir, item)
    if os.path.isfile(item_path):
        file_size = os.path.getsize(item_path) / 1024
        generated_files.append({
            'filename': item,
            'size_kb': round(file_size, 2),
            'path': item_path
        })

end_time = datetime.now()
execution_time = (end_time - start_time).total_seconds()

manifest = {
    'metadata': {
        'dataset_name': dataset_name,
        'generated_at': datetime.now().isoformat(),
        'execution_time_seconds': round(execution_time, 2)
    },
    'dataset': {
        'shape': {'rows': int(df.shape[0]), 'columns': int(df.shape[1])},
        'memory_mb': round(df.memory_usage(deep=True).sum() / 1024**2, 2),
        'columns': df.columns.tolist(),
        'numerical_columns': numerical_cols,
        'categorical_columns': categorical_cols
    },
    'generated_files': generated_files
}

manifest_file = os.path.join(dataset_output_dir, 'manifest.json')
with open(manifest_file, 'w') as f:
    json.dump(manifest, f, indent=2)

print(json.dumps(manifest, indent=2))
print(f'\n✓ Manifest saved: {manifest_file}')
logger.info(f'Manifest exported to {manifest_file}')

## 20. Final Report & Summary

In [ ]:
print('\n' + '='*80)
print('EXECUTION COMPLETE')
print('='*80)

print(f'\n✓ EDA Pipeline finished successfully!')
print(f'Execution Time: {execution_time:.2f} seconds')
print(f'Output Directory: {dataset_output_dir}')
print(f'\nGenerated Files:')
for i, file_info in enumerate(generated_files, 1):
    print(f'  {i}. {file_info["filename"]} ({file_info["size_kb"]} KB)')

print(f'\nLog file: {log_file}')

logger.info('='*80)
logger.info('EDA Pipeline Completed Successfully')
logger.info(f'Total Execution Time: {execution_time:.2f} seconds')
logger.info('='*80)